# Experiment 6 — Automatic Question Paper Categorization (Bloom's Taxonomy)

**Module 4 · Topic: Neural Networks / NLP · CO4 · PBL**

### Problem Statement
> An educational institute wants to classify examination questions according to
> **Bloom's Taxonomy** using deep learning.

### Tasks covered
1. Explain text preprocessing
2. Compare **LSTM** and **Transformer** models
3. Suggest suitable **word embedding** techniques
4. Discuss evaluation metrics
5. Explain possible challenges

### Bloom's Taxonomy (revised, Anderson & Krathwohl 2001)

| Level | Cognitive skill | Typical question verbs |
|---|---|---|
| L1 | **Remember** | define, list, state, name, recall, identify |
| L2 | **Understand** | explain, describe, summarise, interpret, classify |
| L3 | **Apply** | compute, solve, demonstrate, use, implement |
| L4 | **Analyze** | compare, differentiate, examine, categorise, infer |
| L5 | **Evaluate** | justify, critique, assess, defend, recommend |
| L6 | **Create** | design, construct, formulate, propose, develop |

### How to run
`Runtime -> Change runtime type -> **T4 GPU**`, then `Runtime -> Run all` (~10–15 min).

The dataset is **generated inside the notebook** from verb x topic x template combinations, so
the notebook is fully self-contained and reproducible. Swap in a real question bank CSV at
Section 2 if you have one.

## 0. Setup

In [ ]:
import os, re, math, time, random, json, itertools, warnings, collections
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: enable the T4 GPU via Runtime -> Change runtime type")

In [ ]:
# Transformers is preinstalled on Colab; install quietly if missing.
try:
    import transformers
    print("transformers", transformers.__version__)
except ImportError:
    os.system("pip install -q transformers")
    import transformers
    print("installed transformers", transformers.__version__)

## 1. Bloom level definitions and the verb taxonomy

The **action verb** is the single strongest signal of cognitive level — but it is not sufficient
on its own ("*explain how you would design...*" is L6, not L2). Section 9 discusses exactly this
failure mode.

In [ ]:
BLOOM = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]
LABEL2ID = {b: i for i, b in enumerate(BLOOM)}

VERBS = {
    "Remember":   ["define", "list", "state", "name", "recall", "identify", "label", "recognize",
                   "write down", "enumerate", "who invented", "what is", "mention", "reproduce"],
    "Understand": ["explain", "describe", "summarize", "interpret", "illustrate", "discuss",
                   "outline", "paraphrase", "give an example of", "why does", "clarify", "restate"],
    "Apply":      ["compute", "solve", "calculate", "implement", "demonstrate", "apply",
                   "use", "execute", "simulate", "write a program to", "determine", "show how to"],
    "Analyze":    ["compare", "differentiate", "analyze", "examine", "categorize", "contrast",
                   "break down", "infer", "distinguish between", "investigate", "deduce", "trace"],
    "Evaluate":   ["justify", "critique", "assess", "evaluate", "defend", "recommend", "judge",
                   "argue whether", "validate", "prioritize", "appraise", "which is better"],
    "Create":     ["design", "construct", "formulate", "propose", "develop", "devise", "compose",
                   "build", "generate", "plan", "invent", "synthesize a"],
}

TOPICS = [
    "a convolutional neural network", "the backpropagation algorithm", "gradient descent",
    "the transformer architecture", "an autoencoder", "dropout regularization",
    "batch normalization", "a recurrent neural network", "the attention mechanism",
    "transfer learning", "the vanishing gradient problem", "a support vector machine",
    "k-means clustering", "the bias-variance tradeoff", "a decision tree",
    "the TCP three-way handshake", "a binary search tree", "database normalization",
    "the OSI model", "a deadlock in operating systems", "paging and segmentation",
    "the quicksort algorithm", "public key cryptography", "the MapReduce paradigm",
    "a REST API", "the SOLID principles", "dynamic programming", "process scheduling",
    "an ER diagram", "virtual memory", "polymorphism in OOP", "the halting problem",
]

TEMPLATES = [
    "{verb} {topic}.",
    "{verb} {topic} with a suitable example.",
    "{verb} {topic} in your own words.",
    "{verb} {topic} and support your answer with a diagram.",
    "With reference to the syllabus, {verb} {topic}.",
    "{verb} {topic}. [{marks} marks]",
    "Q{qno}) {verb} {topic}.",
    "{verb} {topic} for a real-world scenario.",
    "Briefly, {verb} {topic}.",
    "{verb} {topic} and state its limitations.",
    "In the context of modern systems, {verb} {topic}.",
    "{verb} {topic}; illustrate with a case study.",
]

for lvl, vs in VERBS.items():
    print(f"{lvl:<11} {len(vs)} verbs -> {', '.join(vs[:5])} ...")

## 2. Dataset generation

`n_per_class` questions per Bloom level, combining verb x topic x template, plus deliberate
**realistic noise**: casing mistakes, stray punctuation, mark annotations and question numbers —
so the preprocessing stage in Section 3 actually has work to do.

> **Using your own data instead:** replace this cell with
> `df = pd.read_csv("questions.csv")` where the file has columns `question` and `bloom_level`.

In [ ]:
def generate_dataset(n_per_class=700, seed=SEED):
    rng = random.Random(seed)
    rows = []
    for lvl in BLOOM:
        seen = set()
        made = 0
        guard = 0
        while made < n_per_class and guard < n_per_class * 60:
            guard += 1
            v = rng.choice(VERBS[lvl])
            t = rng.choice(TOPICS)
            tpl = rng.choice(TEMPLATES)
            q = tpl.format(verb=v, topic=t, marks=rng.choice([2, 4, 5, 8, 10]),
                           qno=rng.randint(1, 10))
            q = q[0].upper() + q[1:]
            # realistic noise
            r = rng.random()
            if r < 0.10:  q = q.upper()
            elif r < 0.18: q = q.lower()
            if rng.random() < 0.12: q = q.replace(".", " ..")
            if rng.random() < 0.10: q = "  " + q + "   "
            if rng.random() < 0.08: q = q + f"  (CO{rng.randint(1,6)})"
            if q in seen:  continue
            seen.add(q)
            rows.append((q, lvl))
            made += 1
    rng.shuffle(rows)
    return pd.DataFrame(rows, columns=["question", "bloom_level"])

df = generate_dataset(700)
df["label"] = df["bloom_level"].map(LABEL2ID)
print("dataset:", df.shape)
print(df["bloom_level"].value_counts(), "\n")
df.sample(10, random_state=1)[["question", "bloom_level"]]

## 3. Task 1 — Text preprocessing

The pipeline, and the reason for every step:

| Step | Purpose | Caution for this task |
|---|---|---|
| **Lower-casing** | collapses `EXPLAIN`/`Explain`/`explain` into one token | fine here; harmful for NER |
| **Strip question numbering / mark annotations** (`Q3)`, `[5 marks]`, `(CO2)`) | pure noise, no cognitive signal | must be done before tokenisation |
| **Unicode normalisation + punctuation cleanup** | removes smart quotes, stray `..` | keep the `?` — it correlates with question type |
| **Tokenisation** | splits into lexical units | word-level for the LSTM, **sub-word (WordPiece)** for the Transformer |
| **Stop-word removal** | shrinks vocabulary | **NOT applied here** — `how`, `why`, `which` are exactly the discriminative words |
| **Lemmatisation / stemming** | `designing -> design` maps verb inflections to the base verb | helps the LSTM; skip for BERT, whose sub-word model already handles morphology |
| **Padding / truncation to a fixed length** | enables batching | pad *post*, mask the pads in the loss and in attention |
| **Vocabulary with `<unk>` and `<pad>`** | handles OOV words at test time | min frequency of 2 to drop hapax typos |

In [ ]:
import unicodedata

STOPWORDS_KEEP = {"how", "why", "what", "which", "when", "who", "whether", "not", "no"}

def clean_text(s):
    s = unicodedata.normalize("NFKC", str(s))
    s = s.lower().strip()
    s = re.sub(r"^\s*q\s*\.?\s*\d+\s*[\)\.\:]", " ", s)     # leading "Q3)" / "q.4:"
    s = re.sub(r"\[\s*\d+\s*marks?\s*\]", " ", s)            # "[5 marks]"
    s = re.sub(r"\(\s*co\s*\d\s*\)", " ", s)                 # "(CO2)"
    s = re.sub(r"[^a-z0-9\?\s\-]", " ", s)                   # keep '?' and '-'
    s = re.sub(r"\s+", " ", s).strip()
    return s

LEMMA_RULES = [("ies$", "y"), ("ing$", ""), ("ed$", ""), ("s$", "")]
def simple_lemma(tok):
    if len(tok) <= 4:
        return tok
    for pat, rep in LEMMA_RULES:
        new = re.sub(pat, rep, tok)
        if new != tok and len(new) >= 3:
            return new
    return tok

def tokenize(s, lemmatize=True):
    toks = clean_text(s).split()
    return [simple_lemma(t) if lemmatize else t for t in toks]

for q in df["question"].head(5):
    print("RAW  :", repr(q))
    print("CLEAN:", clean_text(q))
    print("TOKS :", tokenize(q), "\n")

In [ ]:
# Corpus statistics after cleaning
df["clean"] = df["question"].apply(clean_text)
df["tokens"] = df["clean"].apply(lambda s: tokenize(s))
lens = df["tokens"].apply(len)
print(f"token length: mean {lens.mean():.1f} | median {lens.median():.0f} | p95 {np.percentile(lens,95):.0f} | max {lens.max()}")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].hist(lens, bins=25, color="#4c8fbd"); axes[0].set_title("Question length (tokens)")
axes[0].set_xlabel("tokens"); axes[0].grid(alpha=.3)

first_tok = collections.Counter()
for lvl in BLOOM:
    sub = df[df.bloom_level == lvl]["tokens"]
    first_tok[lvl] = collections.Counter(t[0] for t in sub if t).most_common(3)
axes[1].bar(BLOOM, df.groupby("bloom_level").size().reindex(BLOOM).values, color="#7fb069")
axes[1].set_title("Class balance"); axes[1].tick_params(axis="x", rotation=30); axes[1].grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

print("\nMost frequent opening tokens per level (the verb signal):")
for lvl, tops in first_tok.items():
    print(f"  {lvl:<11} {tops}")

## 4. Task 3 — Word embedding techniques

| Technique | Type | Pros | Cons | Verdict for Bloom classification |
|---|---|---|---|---|
| One-hot / Bag-of-Words | sparse, count | trivial, interpretable | no semantics, huge dimension, no order | baseline only |
| TF-IDF | sparse, weighted | strong classical baseline | still order-blind | good sanity check (we fit one in §5) |
| **Word2Vec / GloVe / FastText** | static dense | captures semantic similarity, pretrained on billions of tokens; FastText handles OOV via character n-grams | **one vector per word regardless of context** | used to *initialise* the LSTM embedding |
| **Learned task-specific embedding** | dense, trained | tuned exactly to this label set | needs data, no transfer | our LSTM default (dataset is small but consistent) |
| **Contextual (BERT/RoBERTa/DistilBERT)** | dense, contextual | one vector *per occurrence* — resolves "design" as a noun vs verb; sub-word tokeniser kills OOV | heavier, needs GPU | **best choice**, and exactly why the Transformer wins in §7 |
| Sentence embeddings (SBERT) | dense, sentence-level | one vector per question, great for clustering/retrieval | fixed, less tunable | good for duplicate-question detection |

The critical point for this task: **"design" in _Remember the design of X_ vs _Design a system for X_
is the same static vector but two different Bloom levels.** Only a contextual embedding can tell
them apart.

In [ ]:
# Build the word-level vocabulary for the LSTM
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df,  test_df  = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

PAD, UNK = "<pad>", "<unk>"
counter = collections.Counter(t for toks in train_df["tokens"] for t in toks)
vocab = [PAD, UNK] + [w for w, c in counter.most_common() if c >= 2]
stoi = {w: i for i, w in enumerate(vocab)}
print("vocab size:", len(vocab))
print("top tokens:", counter.most_common(12))

MAX_LEN = int(np.percentile(lens, 98))
print("MAX_LEN:", MAX_LEN)

def encode(tokens, max_len=MAX_LEN):
    ids = [stoi.get(t, 1) for t in tokens][:max_len]
    return ids + [0] * (max_len - len(ids)), min(len(ids), max_len)

class QuestionDS(Dataset):
    def __init__(self, frame):
        self.x, self.l, self.y = [], [], []
        for toks, lab in zip(frame["tokens"], frame["label"]):
            ids, ln = encode(list(toks))
            self.x.append(ids); self.l.append(max(ln, 1)); self.y.append(lab)
        self.x = torch.tensor(self.x); self.l = torch.tensor(self.l); self.y = torch.tensor(self.y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.x[i], self.l[i], self.y[i]

BATCH = 64
tr_dl = DataLoader(QuestionDS(train_df), batch_size=BATCH, shuffle=True)
va_dl = DataLoader(QuestionDS(val_df),   batch_size=256)
te_dl = DataLoader(QuestionDS(test_df),  batch_size=256)

## 5. Classical baseline — TF-IDF + Logistic Regression

Always establish the cheap baseline first. If a linear model on TF-IDF already scores 95 %, the
task is verb-lookup and a neural model is over-engineering; the gap between this and the neural
models tells you how much *context* actually matters.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = tfidf.fit_transform(train_df["clean"]); Xte = tfidf.transform(test_df["clean"])
lr = LogisticRegression(max_iter=2000, C=4.0).fit(Xtr, train_df["label"])
pred_lr = lr.predict(Xte)

acc_lr = accuracy_score(test_df["label"], pred_lr)
f1_lr  = f1_score(test_df["label"], pred_lr, average="macro")
print(f"TF-IDF + LogReg -> accuracy {acc_lr:.4f} | macro-F1 {f1_lr:.4f}\n")
print(classification_report(test_df["label"], pred_lr, target_names=BLOOM, digits=3))

## 6. Model A — BiLSTM classifier

`Embedding -> BiLSTM -> attention pooling -> dropout -> linear`.

Packed sequences ensure the LSTM never consumes the padding, and attention pooling lets the model
weight the informative tokens (the verb) rather than averaging everything.

In [ ]:
class AttnPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.w = nn.Linear(dim, 1)
    def forward(self, h, mask):
        score = self.w(h).squeeze(-1).masked_fill(~mask, -1e4)
        a = torch.softmax(score, dim=1)
        return (h * a.unsqueeze(-1)).sum(1), a

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hid=128, n_classes=6, layers=2, p=0.4):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.drop_emb = nn.Dropout(0.2)
        self.lstm = nn.LSTM(emb_dim, hid, num_layers=layers, batch_first=True,
                            bidirectional=True, dropout=p if layers > 1 else 0.0)
        self.pool = AttnPool(hid * 2)
        self.head = nn.Sequential(nn.Dropout(p), nn.Linear(hid * 2, 128),
                                  nn.ReLU(inplace=True), nn.Dropout(p), nn.Linear(128, n_classes))
    def forward(self, x, lengths, return_attn=False):
        mask = x != 0
        e = self.drop_emb(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        pooled, attn = self.pool(h, mask)
        logits = self.head(pooled)
        return (logits, attn) if return_attn else logits

lstm_model = BiLSTMClassifier(len(vocab))
print(lstm_model)
print("params:", f"{sum(p.numel() for p in lstm_model.parameters()):,}")

In [ ]:
def run_epochs(model, epochs=25, lr=2e-3, tag="BiLSTM"):
    model = model.to(device)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
    best, best_state = 0, None
    for ep in range(1, epochs + 1):
        model.train(); c = n = 0; tot = 0.0
        for x, l, y in tr_dl:
            x, l, y = x.to(device), l.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            out = model(x, l); loss = crit(out, y)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 3.0); opt.step()
            tot += loss.item() * y.size(0); c += (out.argmax(1) == y).sum().item(); n += y.size(0)
        sched.step()
        model.eval(); vc = vn = 0; vtot = 0.0
        with torch.no_grad():
            for x, l, y in va_dl:
                x, l, y = x.to(device), l.to(device), y.to(device)
                out = model(x, l); vtot += crit(out, y).item() * y.size(0)
                vc += (out.argmax(1) == y).sum().item(); vn += y.size(0)
        hist["train_acc"].append(c/n); hist["val_acc"].append(vc/vn)
        hist["train_loss"].append(tot/n); hist["val_loss"].append(vtot/vn)
        if ep % 2 == 0 or ep == 1:
            print(f"[{tag}] ep {ep:02d}/{epochs} | train {c/n:.4f} | val {vc/vn:.4f} loss {vtot/vn:.4f}")
        if vc/vn > best:
            best = vc/vn
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f"[{tag}] best val acc {best:.4f}")
    return model, hist

lstm_model, hist_lstm = run_epochs(BiLSTMClassifier(len(vocab)), epochs=25)

@torch.no_grad()
def eval_lstm(model, loader):
    model.eval(); P, Y = [], []
    for x, l, y in loader:
        P.append(model(x.to(device), l.to(device)).argmax(1).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

pred_lstm, y_test = eval_lstm(lstm_model, te_dl)
acc_lstm = accuracy_score(y_test, pred_lstm); f1_lstm = f1_score(y_test, pred_lstm, average="macro")
print(f"\nBiLSTM -> test accuracy {acc_lstm:.4f} | macro-F1 {f1_lstm:.4f}")

In [ ]:
# What is the attention head looking at?
lstm_model.eval()
samples = test_df.sample(6, random_state=3)
for _, row in samples.iterrows():
    toks = list(row["tokens"])[:MAX_LEN]
    ids, ln = encode(toks)
    x = torch.tensor([ids]).to(device); l = torch.tensor([max(ln, 1)]).to(device)
    with torch.no_grad():
        logits, a = lstm_model(x, l, return_attn=True)
    a = a[0, :len(toks)].cpu().numpy()
    top = np.argsort(a)[::-1][:3]
    pred = BLOOM[logits.argmax(1).item()]
    hl = " ".join(f"**{t}**" if i in top else t for i, t in enumerate(toks))
    print(f"true={row['bloom_level']:<11} pred={pred:<11} | {hl}")

## 7. Model B — Transformer

Two variants:

* **B1 — Transformer encoder trained from scratch** (self-attention + positional encoding),
  so the mechanism is fully visible.
* **B2 — Fine-tuned DistilBERT**, the pretrained contextual model. This is what you would
  actually deploy.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, p=0.1):
        super().__init__()
        self.drop = nn.Dropout(p)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, layers=3, ff=256, n_classes=6, p=0.2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.scale = math.sqrt(d_model)
        self.pos = PositionalEncoding(d_model, p=p)
        enc = nn.TransformerEncoderLayer(d_model, nhead, ff, dropout=p,
                                         batch_first=True, activation="gelu", norm_first=True)
        self.enc = nn.TransformerEncoder(enc, layers)
        self.pool = AttnPool(d_model)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(p), nn.Linear(d_model, n_classes))
    def forward(self, x, lengths=None, return_attn=False):
        mask = x != 0
        h = self.pos(self.emb(x) * self.scale)
        h = self.enc(h, src_key_padding_mask=~mask)
        pooled, a = self.pool(h, mask)
        logits = self.head(pooled)
        return (logits, a) if return_attn else logits

tr_model, hist_tr = run_epochs(TransformerClassifier(len(vocab)), epochs=25, lr=1e-3, tag="Transformer")
pred_tr, _ = eval_lstm(tr_model, te_dl)
acc_tr = accuracy_score(y_test, pred_tr); f1_tr = f1_score(y_test, pred_tr, average="macro")
print(f"\nTransformer (scratch) -> test accuracy {acc_tr:.4f} | macro-F1 {f1_tr:.4f}")

### 7.2 Fine-tuning DistilBERT (contextual pretrained embeddings)

In [ ]:
from torch.utils.data import TensorDataset

bert_ok = True
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    MODEL_NAME = "distilbert-base-uncased"
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    bert = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=6,
        id2label={i: b for i, b in enumerate(BLOOM)}, label2id=LABEL2ID).to(device)
    print("loaded", MODEL_NAME, f"({sum(p.numel() for p in bert.parameters()):,} params)")
except Exception as e:
    bert_ok = False
    print("DistilBERT unavailable (offline?) — skipping section 7.2.\n", repr(e)[:250])

In [ ]:
if bert_ok:
    def make_bert_ds(frame):
        enc = tok(list(frame["question"].astype(str)), truncation=True, padding="max_length",
                  max_length=48, return_tensors="pt")
        return TensorDataset(enc["input_ids"], enc["attention_mask"],
                             torch.tensor(frame["label"].values))

    b_tr = DataLoader(make_bert_ds(train_df), batch_size=32, shuffle=True)
    b_va = DataLoader(make_bert_ds(val_df),   batch_size=64)
    b_te = DataLoader(make_bert_ds(test_df),  batch_size=64)

    opt = torch.optim.AdamW(bert.parameters(), lr=3e-5, weight_decay=0.01)
    EPOCHS_B = 3
    total_steps = EPOCHS_B * len(b_tr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-5, total_steps=total_steps, pct_start=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    def bert_eval(loader):
        bert.eval(); P, Y = [], []
        with torch.no_grad():
            for ids, am, y in loader:
                with torch.autocast("cuda", enabled=(device.type == "cuda")):
                    out = bert(input_ids=ids.to(device), attention_mask=am.to(device)).logits
                P.append(out.argmax(1).cpu()); Y.append(y)
        return torch.cat(P).numpy(), torch.cat(Y).numpy()

    for ep in range(1, EPOCHS_B + 1):
        bert.train(); c = n = 0
        t0 = time.time()
        for ids, am, y in b_tr:
            ids, am, y = ids.to(device), am.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=(device.type == "cuda")):
                out = bert(input_ids=ids, attention_mask=am, labels=y)
            scaler.scale(out.loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            c += (out.logits.argmax(1) == y).sum().item(); n += y.size(0)
        pv, yv = bert_eval(b_va)
        print(f"[DistilBERT] ep {ep}/{EPOCHS_B} | train {c/n:.4f} | val {accuracy_score(yv, pv):.4f} "
              f"| {time.time()-t0:.0f}s")

    pred_bert, y_b = bert_eval(b_te)
    acc_bert = accuracy_score(y_b, pred_bert); f1_bert = f1_score(y_b, pred_bert, average="macro")
    print(f"\nDistilBERT -> test accuracy {acc_bert:.4f} | macro-F1 {f1_bert:.4f}")
else:
    acc_bert = f1_bert = float("nan"); pred_bert = None

## 8. Task 2 — LSTM vs Transformer, and Task 4 — evaluation metrics

### Architectural comparison

| Dimension | **LSTM / BiLSTM** | **Transformer** |
|---|---|---|
| Sequence processing | **Sequential** — step *t* needs step *t-1* | **Parallel** — all positions at once |
| Long-range dependency | Decays with distance; gating helps but does not eliminate the problem | **O(1) path length** between any two tokens via self-attention |
| Complexity per layer | O(n · d²) | O(n² · d) — worse for long sequences, better for short ones like exam questions |
| GPU utilisation | Poor (serial recurrence) | Excellent (one big matmul) — the reason it wins on a T4 |
| Parameters (here) | ~0.6 M | ~0.8 M scratch / 67 M DistilBERT |
| Data hunger | Works with a few thousand examples | From scratch it needs more data; **pretraining removes that constraint** |
| Interpretability | Attention pooling weights | Per-head, per-layer attention maps — much richer |
| Positional information | Implicit in the recurrence | Must be **injected** (sinusoidal or learned) |
| Best for this task | Short questions, tiny dataset, cheap CPU deployment | The accurate option; DistilBERT's contextual embeddings resolve verb ambiguity |

### Evaluation metrics for Bloom classification

* **Macro-F1** — the headline number. The class prior of a real question bank is skewed towards
  L1/L2, so plain accuracy flatters a model that ignores L5/L6.
* **Per-class precision/recall** — an institute cares most about correctly identifying the
  *higher-order* levels (L4–L6) for NBA/NAAC outcome-attainment reports.
* **Confusion matrix** — the errors should be *adjacent* (L2↔L3), never distant (L1↔L6).
* **Quadratic Weighted Kappa (QWK)** — Bloom levels are **ordinal**, so predicting L5 for an L6
  question is a smaller error than predicting L1. QWK is the correct metric and is reported below.
* **Mean Absolute Error in levels** — interpretable ordinal error ("off by 0.3 levels on average").
* **Cohen's kappa against a second human annotator** — the human ceiling; inter-rater agreement on
  Bloom labelling is typically only 0.6–0.7, which bounds what any model can achieve.
* **Calibration / confidence** — low-confidence questions should be routed to a human reviewer.

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix

rows = [("TF-IDF + LogReg", acc_lr, f1_lr, pred_lr),
        ("BiLSTM", acc_lstm, f1_lstm, pred_lstm),
        ("Transformer (scratch)", acc_tr, f1_tr, pred_tr)]
if bert_ok:
    rows.append(("DistilBERT (fine-tuned)", acc_bert, f1_bert, pred_bert))

print(f"{'model':<26}{'acc':>8}{'macro-F1':>11}{'QWK':>8}{'MAE(levels)':>13}")
print("-" * 66)
summary = {}
for name, a, f, p in rows:
    qwk = cohen_kappa_score(y_test, p, weights="quadratic")
    mae = np.abs(np.array(p) - y_test).mean()
    summary[name] = dict(acc=a, f1=f, qwk=qwk, mae=mae)
    print(f"{name:<26}{a:>8.4f}{f:>11.4f}{qwk:>8.4f}{mae:>13.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].plot(hist_lstm["val_acc"], marker="o", ms=3, label="BiLSTM val")
axes[0].plot(hist_tr["val_acc"],   marker="s", ms=3, label="Transformer val")
axes[0].plot(hist_lstm["train_acc"], "--", alpha=.5, label="BiLSTM train")
axes[0].plot(hist_tr["train_acc"],   "--", alpha=.5, label="Transformer train")
axes[0].set_xlabel("epoch"); axes[0].set_title("Learning curves"); axes[0].legend(); axes[0].grid(alpha=.3)

names = list(summary); x = np.arange(len(names)); wdt = 0.27
axes[1].bar(x - wdt, [summary[n]["acc"] for n in names], wdt, label="accuracy")
axes[1].bar(x,       [summary[n]["f1"]  for n in names], wdt, label="macro-F1")
axes[1].bar(x + wdt, [summary[n]["qwk"] for n in names], wdt, label="QWK")
axes[1].set_xticks(x); axes[1].set_xticklabels([n.split(" (")[0] for n in names], rotation=18, fontsize=8)
axes[1].set_ylim(0, 1.05); axes[1].legend(); axes[1].grid(axis="y", alpha=.3); axes[1].set_title("Model comparison")
plt.tight_layout(); plt.show()

In [ ]:
best_name, best_pred = max(rows, key=lambda r: r[2])[0], max(rows, key=lambda r: r[2])[3]
cm = confusion_matrix(y_test, best_pred)
cmn = cm / cm.sum(1, keepdims=True)

fig, ax = plt.subplots(figsize=(6.4, 5.6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(6)); ax.set_xticklabels(BLOOM, rotation=40, ha="right")
ax.set_yticks(range(6)); ax.set_yticklabels(BLOOM)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if cmn[i, j] > .5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"Confusion matrix — {best_name}")
fig.colorbar(im); plt.tight_layout(); plt.show()

print("Off-diagonal mass by level distance (adjacent errors are acceptable, distant ones are not):")
d = np.abs(np.subtract.outer(range(6), range(6)))
for k in range(1, 6):
    print(f"  distance {k}: {cm[d == k].sum():4d} errors")

## 9. Task 5 — Challenges, and a live demo on adversarial questions

In [ ]:
# Questions engineered to break a verb-lookup heuristic
tricky = [
    ("Explain how you would design a new caching scheme for a CDN.", "Create"),
    ("List the steps you would follow to evaluate two competing architectures.", "Evaluate"),
    ("Define the term 'analysis' as used in software engineering.", "Remember"),
    ("Compare the given code with the specification and state which is correct.", "Analyze"),
    ("Design a table listing the five OSI layers.", "Remember"),
    ("Apply the concept of entropy to justify your choice of split in a decision tree.", "Evaluate"),
]

def predict_all(text):
    out = {}
    out["TF-IDF"] = BLOOM[lr.predict(tfidf.transform([clean_text(text)]))[0]]
    ids, ln = encode(tokenize(text))
    x = torch.tensor([ids]).to(device); l = torch.tensor([max(ln, 1)]).to(device)
    with torch.no_grad():
        out["BiLSTM"] = BLOOM[lstm_model(x, l).argmax(1).item()]
        out["Transformer"] = BLOOM[tr_model(x, l).argmax(1).item()]
        if bert_ok:
            e = tok([text], truncation=True, padding="max_length", max_length=48, return_tensors="pt")
            lo = bert(input_ids=e["input_ids"].to(device), attention_mask=e["attention_mask"].to(device)).logits
            out["DistilBERT"] = BLOOM[lo.argmax(1).item()]
    return out

print(f"{'question':<72}{'gold':<11}{'predictions'}")
print("-" * 130)
for q, gold in tricky:
    p = predict_all(q)
    flags = " | ".join(f"{k}:{v}" for k, v in p.items())
    print(f"{q[:70]:<72}{gold:<11}{flags}")

### Challenges in deploying this at an institute

| Challenge | Detail | Mitigation |
|---|---|---|
| **Label subjectivity** | Two faculty members disagree on L2 vs L3 for the same question; inter-rater kappa is often only 0.6–0.7. | Multi-annotator labelling, adjudication, train on the consensus label, and report QWK against the human ceiling rather than pretending 100 % is reachable. |
| **Verb ambiguity** | "Explain how you would *design*…" is L6, not L2. The demo above is exactly this trap. | Contextual embeddings (DistilBERT), plus syntactic parsing to find the *main* clause verb. |
| **Class imbalance** | Real question banks are 60 % L1/L2, with very few L6. | Class-weighted loss, focal loss, oversampling, back-translation augmentation of the rare levels. |
| **Domain shift** | A model trained on CS questions misreads Mechanical or Management questions. | Per-department fine-tuning, domain-adaptive pretraining on the institute's own corpus. |
| **Short, context-free text** | Many questions are 5–8 tokens — very little signal. | Concatenate the CO/unit metadata and marks allotted as extra features. |
| **Multilingual / code-mixed papers** | Regional language papers, or English questions containing code snippets and formulae. | mBERT/IndicBERT/XLM-R; strip or specially tokenise LaTeX and code blocks. |
| **Ordinality ignored** | A standard cross-entropy model treats L1↔L6 as no worse than L1↔L2. | Ordinal regression loss (CORAL), or soft labels over neighbouring levels. |
| **Trust and adoption** | Faculty will not accept a black box that governs NBA attainment reports. | Attention-highlighted explanations (Section 6), confidence thresholding, human-in-the-loop review of low-confidence questions. |
| **Data privacy** | Unreleased question papers are confidential. | On-premise inference, no third-party API calls, access logging. |
| **Small data** | A department may have only a few hundred labelled questions. | Transfer learning + few-shot fine-tuning; this is decisive in favour of the pretrained Transformer. |

## 10. Conclusion

* **Preprocessing matters but must be selective** — removing stop-words here would destroy the
  `how / why / which` signal that separates L2 from L4.
* The **TF-IDF baseline is strong** on template-generated data because the verb is nearly a
  giveaway; its failures on the adversarial set in Section 9 are precisely where deep models earn
  their keep.
* The **BiLSTM** reads the question sequentially and its attention head lands on the action verb;
  it is the cheapest neural option and deploys on CPU.
* The **Transformer** trains in parallel, reaches every token in one hop, and — once
  **pretrained (DistilBERT)** — wins on both macro-F1 and QWK because its embeddings are
  *contextual*, resolving the "design as noun vs verb" ambiguity a static embedding cannot.
* Because Bloom levels are **ordinal**, QWK and MAE-in-levels are the metrics to report, not raw
  accuracy; the confusion matrix confirms the residual errors are almost entirely between
  *adjacent* levels, which is the acceptable failure mode.

**Production recommendation:** fine-tune DistilBERT on the institute's own labelled bank, apply
an ordinal (CORAL) loss, threshold on confidence, and route the bottom 10 % to a human reviewer.